# BTL02 - FER2013 Model Training on Google Colab (GPU T4)
Notebook này dùng để huấn luyện tự động 3 mô hình Custom CNN (SmallCNN, MediumCNN, LargeCNN) trên GPU miễn phí của Google Colab.
**LƯU Ý QUAN TRỌNG:** Hãy vào menu **Runtime** -> **Change runtime type** -> chọn **T4 GPU** trước khi chạy!


In [8]:
# 1. Kéo mã nguồn dự án từ nhánh của nhóm về Colab
import os
import sys
import shutil

repo_name = 'Deep_Learning_Group'
repo_url = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
branch_name = 'feat/BTL02_T0103_T0104_T0105_T0201_T0202_data_and_cnn_models'

# LƯU Ý: Vui lòng nhập GitHub Personal Access Token (PAT) của bạn vào đây nếu đây là private repository
github_token = ""  # Ví dụ: "ghp_abc123XYZ..."

# Đảm bảo dọn dẹp thư mục lỗi trước đó nếu có
if os.path.exists(repo_name) and not os.path.exists(os.path.join(repo_name, 'BTL02')):
    print(f"Dọn dẹp thư mục lỗi cũ: {repo_name}")
    shutil.rmtree(repo_name, ignore_errors=True)

if github_token:
    # Sử dụng Token để xác thực clone private repo
    repo_url = f'https://{github_token}@github.com/Duy-DS/Deep_Learning_Group.git'
    print("Sử dụng Personal Access Token để xác thực...")
else:
    print("⚠️ CẢNH BÁO: Chưa cung cấp github_token. Nếu đây là repository riêng tư (private), việc clone sẽ thất bại.")

if not os.path.exists(repo_name):
    print(f"Đang clone repository từ nhánh {branch_name}...")
    !git clone -b {branch_name} {repo_url}

# Di chuyển vào thư mục BTL02 của dự án
btl_dir = os.path.abspath(os.path.join(os.getcwd(), repo_name, 'BTL02'))
if os.path.exists(btl_dir):
    os.chdir(btl_dir)
    print(f"Đã di chuyển thư mục làm việc đến: {os.getcwd()}")
else:
    print(f"❌ Thất bại: Không tìm thấy thư mục {btl_dir}.")
    print("Vui lòng kiểm tra lại tính chính xác của github_token hoặc quyền truy cập của tài khoản của bạn vào repository này.")

# Thêm thư mục BTL02 vào sys.path để Python định vị được package 'src'
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())


Sử dụng Personal Access Token để xác thực...
Đang clone repository từ nhánh feat/BTL02_T0103_T0104_T0105_T0201_T0202_data_and_cnn_models...
Cloning into 'Deep_Learning_Group'...
remote: Enumerating objects: 1653, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 1653 (delta 39), reused 73 (delta 29), pack-reused 1543 (from 3)
Receiving objects: 100% (1653/1653), 236.20 MiB | 18.95 MiB/s, done.
Resolving deltas: 100% (611/611), done.
Đã di chuyển thư mục làm việc đến: /content/Deep_Learning_Group/BTL02


In [9]:
# 2. Cài đặt các thư viện cần thiết cho MLOps
!pip install -q mlflow pyyaml


In [10]:
# 3. Tải và giải nén bộ dữ liệu chuẩn fer2013.csv (~96MB nén)
import urllib.request
import zipfile
import os

os.makedirs('data/raw', exist_ok=True)
csv_target = 'data/raw/fer2013.csv'
zip_target = 'data/raw/fer2013.csv.zip'
dataset_url = 'https://huggingface.co/datasets/chitradrishti/fer2013/resolve/main/fer2013.csv.zip'

if not os.path.exists(csv_target):
    print('Đang tải dữ liệu fer2013.csv.zip...')
    urllib.request.urlretrieve(dataset_url, zip_target)
    print('Đang giải nén...')
    with zipfile.ZipFile(zip_target, 'r') as z:
        z.extractall('data/raw')
    if os.path.exists(zip_target):
        os.remove(zip_target)
    print('Tải và giải nén hoàn tất! Kích thước file:', os.path.getsize(csv_target), 'bytes')
else:
    print('File dữ liệu đã tồn tại sẵn.')


Đang tải dữ liệu fer2013.csv.zip...
Đang giải nén...
Tải và giải nén hoàn tất! Kích thước file: 301072766 bytes


In [11]:
# 4. Kiểm tra GPU
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))


PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4


In [12]:
# 5. Huấn luyện Baseline SmallCNN (EXP-01 - 2 blocks Conv)
import os
from src.train.trainer import Trainer

print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN EXP-01 (SmallCNN)...')
print('=' * 60)

# Đảm bảo thư mục đầu ra tồn tại trước khi chạy
os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/logs', exist_ok=True)

trainer_small = Trainer(config='experiments/baseline/exp01_small_cnn.yaml')
res_small = trainer_small.fit()

🚀 BẮT ĐẦU HUẤN LUYỆN EXP-01 (SmallCNN)...


2026/11/17 11:34:25 INFO mlflow.tracking.fluent: Experiment with name 'BTL02_FER2013_PyTorch' does not exist. Creating a new experiment.


[*] Khởi tạo thành công mô hình SmallCNN: 1,199,495 tham số.

🚀 BẮT ĐẦU THỰC NGHIỆM: EXP-01_SmallCNN (Nhóm: Baseline | Câu hỏi: RQ1)
Epoch [01/50] Train Loss: 1.6415 | Train Acc: 34.98% | Val Loss: 1.4790 | Val Acc: 42.96% | Val F1: 0.3292
  --> [*] Lưu best checkpoint tại Epoch 1 (Val Acc: 42.96%)
Epoch [02/50] Train Loss: 1.4631 | Train Acc: 43.34% | Val Loss: 1.3783 | Val Acc: 46.92% | Val F1: 0.3649
  --> [*] Lưu best checkpoint tại Epoch 2 (Val Acc: 46.92%)
Epoch [03/50] Train Loss: 1.3732 | Train Acc: 47.35% | Val Loss: 1.3053 | Val Acc: 49.65% | Val F1: 0.3998
  --> [*] Lưu best checkpoint tại Epoch 3 (Val Acc: 49.65%)
Epoch [04/50] Train Loss: 1.3017 | Train Acc: 50.16% | Val Loss: 1.2808 | Val Acc: 50.96% | Val F1: 0.4074
  --> [*] Lưu best checkpoint tại Epoch 4 (Val Acc: 50.96%)
Epoch [05/50] Train Loss: 1.2322 | Train Acc: 53.19% | Val Loss: 1.2385 | Val Acc: 53.00% | Val F1: 0.4364
  --> [*] Lưu best checkpoint tại Epoch 5 (Val Acc: 53.00%)
Epoch [06/50] Train Loss: 1.1762

In [13]:
# 6. Huấn luyện MediumCNN (EXP-02 - 3 blocks Conv)
import os
from src.train.trainer import Trainer

print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN EXP-02 (MediumCNN)...')
print('=' * 60)

trainer_medium = Trainer(config='experiments/baseline/exp02_medium_cnn.yaml')
res_medium = trainer_medium.fit()

🚀 BẮT ĐẦU HUẤN LUYỆN EXP-02 (MediumCNN)...
[*] Khởi tạo thành công mô hình MediumCNN: 2,731,015 tham số.

🚀 BẮT ĐẦU THỰC NGHIỆM: EXP-02_MediumCNN (Nhóm: Custom_CNN | Câu hỏi: RQ1)
Epoch [01/50] Train Loss: 1.6137 | Train Acc: 36.15% | Val Loss: 1.4122 | Val Acc: 46.25% | Val F1: 0.3633
  --> [*] Lưu best checkpoint tại Epoch 1 (Val Acc: 46.25%)
Epoch [02/50] Train Loss: 1.3509 | Train Acc: 48.33% | Val Loss: 1.2582 | Val Acc: 51.32% | Val F1: 0.4103
  --> [*] Lưu best checkpoint tại Epoch 2 (Val Acc: 51.32%)
Epoch [03/50] Train Loss: 1.2214 | Train Acc: 53.77% | Val Loss: 1.2039 | Val Acc: 53.78% | Val F1: 0.4489
  --> [*] Lưu best checkpoint tại Epoch 3 (Val Acc: 53.78%)
Epoch [04/50] Train Loss: 1.1188 | Train Acc: 57.85% | Val Loss: 1.1486 | Val Acc: 54.78% | Val F1: 0.4724
  --> [*] Lưu best checkpoint tại Epoch 4 (Val Acc: 54.78%)
Epoch [05/50] Train Loss: 1.0321 | Train Acc: 61.04% | Val Loss: 1.1591 | Val Acc: 56.48% | Val F1: 0.5243
  --> [*] Lưu best checkpoint tại Epoch 5 (Va

In [14]:
# 7. Huấn luyện LargeCNN (EXP-03 - 4 blocks + BatchNorm + Dropout)
import os
from src.train.trainer import Trainer

print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN EXP-03 (LargeCNN)...')
print('=' * 60)

trainer_large = Trainer(config='experiments/baseline/exp03_large_cnn.yaml')
res_large = trainer_large.fit()

🚀 BẮT ĐẦU HUẤN LUYỆN EXP-03 (LargeCNN)...
[*] Khởi tạo thành công mô hình LargeCNN: 3,916,167 tham số.

🚀 BẮT ĐẦU THỰC NGHIỆM: EXP-03_LargeCNN (Nhóm: Custom_CNN | Câu hỏi: RQ1)
Epoch [01/50] Train Loss: 1.6538 | Train Acc: 34.95% | Val Loss: 1.3871 | Val Acc: 45.81% | Val F1: 0.3452
  --> [*] Lưu best checkpoint tại Epoch 1 (Val Acc: 45.81%)
Epoch [02/50] Train Loss: 1.3557 | Train Acc: 47.62% | Val Loss: 1.2416 | Val Acc: 50.82% | Val F1: 0.4040
  --> [*] Lưu best checkpoint tại Epoch 2 (Val Acc: 50.82%)
Epoch [03/50] Train Loss: 1.2527 | Train Acc: 51.93% | Val Loss: 1.1991 | Val Acc: 53.72% | Val F1: 0.4814
  --> [*] Lưu best checkpoint tại Epoch 3 (Val Acc: 53.72%)
Epoch [04/50] Train Loss: 1.1834 | Train Acc: 54.74% | Val Loss: 1.1409 | Val Acc: 56.34% | Val F1: 0.5148
  --> [*] Lưu best checkpoint tại Epoch 4 (Val Acc: 56.34%)
Epoch [05/50] Train Loss: 1.1337 | Train Acc: 57.00% | Val Loss: 1.0944 | Val Acc: 57.98% | Val F1: 0.5352
  --> [*] Lưu best checkpoint tại Epoch 5 (Val A

In [17]:
import os
import pandas as pd
from google.colab import files

log_file = 'outputs/logs/mlflow_summary_24col.csv'
if os.path.exists(log_file):
    summary_df = pd.read_csv(log_file)
    print('=== TỔNG HỢP CẢ 3 THỰC NGHIỆM TRONG MLFLOW ===')
    # Sử dụng các tên cột chính xác (snake_case) từ MLflow export DataFrame
    display(summary_df[['run_id', 'params.architecture', 'metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1', 'metrics.best_epoch']])
    files.download(log_file)
else:
    print(f'Không tìm thấy tệp log: {log_file}')

checkpoints = [
    'outputs/checkpoints/best_small_cnn.pth',
    'outputs/checkpoints/best_medium_cnn.pth',
    'outputs/checkpoints/best_large_cnn.pth'
]

for cp in checkpoints:
    if os.path.exists(cp):
        print(f'Đang tải về checkpoint: {cp}')
        files.download(cp)
    else:
        print(f'Không tìm thấy checkpoint: {cp}')

=== TỔNG HỢP CẢ 3 THỰC NGHIỆM TRONG MLFLOW ===


,run_id,params.architecture,metrics.best_val_acc,metrics.private_test_acc,metrics.macro_f1,metrics.best_epoch
0,5a71e5b71f034b219810a1bdd240f31d,LargeCNN,63.833937,64.948454,0.635647,32.0
1,1f46b0e87c324a63b35b5385edd3e84d,MediumCNN,59.598774,58.094177,0.570799,20.0
2,49137a591d16416984e43424063d4a13,SmallCNN,55.363611,55.168571,0.493569,8.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đang tải về checkpoint: outputs/checkpoints/best_small_cnn.pth


<IPython.core.display.Javascript object>

... (3 outputs left out)

<IPython.core.display.Javascript object>

Đang tải về checkpoint: outputs/checkpoints/best_large_cnn.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>